# Workshop 1.3: Grouping and Aggregating Data

Welcome back! In our earlier sessions, we learned how to construct DataFrames, access columns, and filter rows using logical rules. While inspecting individual observations is useful, financial analysis usually requires understanding aggregate patterns.

### Summarizing Data by Category

Raw data rows rarely tell the complete story on their own. In portfolio management, we almost always need categorical summaries: What was our total return by sector? What is the average trading volume for each stock? How do risk metrics vary across asset classes?

If you have ever built a Pivot Table in a spreadsheet, you already grasp this concept. In pandas, this categorical grouping is powered by the versatile **`.groupby()`** method.

Under the hood, `.groupby()` executes the classic **split-apply-combine** workflow:
- **Split**: Slices your table into separate partitions based on category labels.
- **Apply**: Computes an aggregation metric, such as a sum or mean, across each partition.
- **Combine**: Assembles the computed answers into a unified summary table.

Let's see this workflow in action. Let's see:

## Topic 1: Creating a Grouped DataFrame

To explore grouping, let's assemble a sales table tracking sales regions alongside product revenue figures.

Let's build our dataset and inspect its structure. Let's see:

In [1]:
import pandas as pd

# Sales data across regions and products:
data = {
  "Region": ["North", "South", "North", "South", "North", "South"],
  "Product": ["Laptop", "Laptop", "Monitor", "Monitor", "Keyboard", "Keyboard"],
  "Sales": [500, 300, 200, 250, 100, 150]
}

df = pd.DataFrame(data)
print(df)

  Region   Product  Sales
0  North    Laptop    500
1  South    Laptop    300
2  North   Monitor    200
3  South   Monitor    250
4  North  Keyboard    100
5  South  Keyboard    150


### Grouping by a Single Column

To partition our table by geographical area, we call `.groupby("Region")`.

Let's see what happens when we run this method. Let's check:

In [2]:
# Group the DataFrame by 'Region':
grouped = df.groupby("Region")

print(grouped)

### Understanding Lazy Evaluation in GroupBy

Notice that pandas did not immediately display a table! Instead, it printed a `DataFrameGroupBy` object. This behavior often surprises beginners, but it is actually a clever design feature known as **lazy evaluation**.

Pandas has organized the rows into internal buckets in memory, but it is patiently waiting for your next instruction: *"What calculation do you want to run on each bucket?"*

> **Key Takeaway**: Calling `.groupby()` prepares partitions in memory without computing values until you supply an aggregation function.

## Topic 2: Applying Aggregate Functions

An **aggregate function** condenses multiple numerical observations into a single summary figure. Once we have a grouped object, we pick our target column and apply a calculation:
- **`.sum()`**: Adds up all values within each partition.
- **`.mean()`**: Computes the numerical average for each group.
- **`.count()`**: Counts the total observations present in each category.

Let's compute total sales alongside regional averages. Let's see:


In [3]:
# We compute total sales per region:
print("--- Total Sales per Region (.sum()) ---")
print(grouped["Sales"].sum())

# We compute average sales per region:
print("\n--- Average Sales per Region (.mean()) ---")
print(grouped["Sales"].mean())

# We count transaction volume per region:
print("\n--- Row Count per Region (.count()) ---")
print(grouped["Sales"].count())

--- Total Sales per Region (.sum()) ---
Region
North    800
South    700
Name: Sales, dtype: int64

--- Average Sales per Region (.mean()) ---
Region
North    266.666667
South    233.333333
Name: Sales, dtype: float64

--- Row Count per Region (.count()) ---
Region
North    3
South    3
Name: Sales, dtype: int64


> **Key Takeaway**: We extract a target column from our grouped object and attach aggregate functions like `.sum()` or `.mean()` to compute group metrics.

---

## Topic 3: Grouping by Multiple Columns

What if we need finer granularity, such as calculating total sales for each specific product within each region? We can pass a list of column names into `.groupby(["Region", "Product"])`.

When we group across multiple columns, pandas forms a **MultiIndex**, which is a hierarchical index with multiple levels of labels.

Let's see how pandas formats a multi-level grouping. Let's check:

In [4]:
# Group by Region first, then by Product inside each region:
multi_grouped = df.groupby(["Region", "Product"])["Sales"].sum()

print(multi_grouped)

Region  Product 
North   Keyboard    100
        Laptop      500
        Monitor     200
South   Keyboard    150
        Laptop      300
        Monitor     250
Name: Sales, dtype: int64


Notice the clear hierarchy on the left: `Region` acts as the primary outer level, while `Product` sits inside as a nested sub-level.

> **Key Takeaway**: Grouping by multiple columns generates a hierarchical MultiIndex that organizes data across nested categorical layers.

## Topic 4: The .agg() Method for Simultaneous Metrics

In financial reporting, we often want to inspect multiple summary metrics side by side, such as total volume alongside average transaction size.

Instead of running separate calculations and manually merging them, pandas offers the **`.agg()`** method. We simply pass a list of function names as strings, such as `["sum", "mean", "count"]`.

Let's generate a complete multi-metric summary in a single call. Let's see:

In [5]:
# We compute multiple metrics simultaneously across our groups:
sales_summary = df.groupby("Region")["Sales"].agg(["sum", "mean", "count"])

print(sales_summary)

        sum        mean  count
Region                        
North   800  266.666667      3
South   700  233.333333      3


> **Key Takeaway**: The `.agg()` method calculates multiple summary statistics simultaneously in a clean, unified table.

---

## Topic 5: Flattening Tables with .reset_index()

Notice that in our outputs above, `Region` was moved into the bold index column on the left. While this format is convenient for reading, downstream tasks like charting or exporting to CSV often require standard columns.

We can turn the group labels back into standard flat columns by appending **`.reset_index()`**.

Let's see how `.reset_index()` restores our table to a flat format. Let's check:

In [6]:
# Calculate sum and reset the index to flatten the table:
result = df.groupby("Region")["Sales"].sum().reset_index()

print(result)

  Region  Sales
0  North    800
1  South    700


### How Quants Use GroupBy in Daily Practice

In quantitative research, `.groupby()` forms the foundation for many core workflows:
- **Grouping by Ticker**: We partition daily market feeds by stock symbol to compute individual asset volatility or rolling momentum signals.
- **Grouping by Sector**: We measure which industry groups lead market trends.
- **Grouping by Month or Quarter**: We analyze seasonal patterns and evaluate monthly portfolio returns.

> **Key Takeaway**: Calling `.reset_index()` flattens grouped results back into standard columns, ready for charting or file export.

---

## Practice Time

Now it is your turn to practice grouping and aggregating datasets. Grouping data is a skill you will use in nearly every quantitative strategy, so work through these challenges deliberately.

---

### Challenge 1: Grouping by Category

- Construct a DataFrame from this dictionary:
  ```python
  product_data = {
      "Category": ["Electronics", "Electronics", "Furniture", "Furniture"],
      "Product": ["Phone", "Headphones", "Desk", "Chair"],
      "Sales": [800, 200, 450, 150]
  }
  ```
- Group by `"Category"` and calculate total sales using `.sum()`.
- Display the resulting series.

In [ ]:
# Challenge 1: Write your code below this line


# Expected Output:
# Category
# Electronics  1000
# Furniture    600
# Name: Sales, dtype: int64


### Challenge 2: Multi-Level Grouping

- Using our original sales DataFrame `df`:
- Group by both `"Region"` and `"Product"`.
- Calculate the **average (mean)** sales for each combination using `.mean()`.
- Display the multi-index result.

In [ ]:
# Challenge 2: Write your code below this line


# Expected Output:
# Region Product
# North  Keyboard  100.0
#     Laptop   500.0
#     Monitor   200.0
# South  Keyboard  150.0
#     Laptop   300.0
#     Monitor   250.0
# Name: Sales, dtype: float64


### Challenge 3: Multi-Metric Aggregation with .agg()

- Using `df`:
- Group by `"Region"`.
- Use `.agg()` to compute both `"sum"` and `"mean"` across the `"Sales"` column.
- Display the resulting summary table.

In [ ]:
# Challenge 3: Write your code below this line


# Expected Output:
#     sum    mean
# Region
# North  800 266.666667
# South  700 233.333333


---

## Solutions Section

Great work tackling these aggregation challenges! Grouping and summarizing records accurately is essential for financial reporting.

Let's review the reference implementations together.

### Reference Code

#### Solution for Challenge 1:
```python
product_data = {
    "Category": ["Electronics", "Electronics", "Furniture", "Furniture"],
    "Product": ["Phone", "Headphones", "Desk", "Chair"],
    "Sales": [800, 200, 450, 150]
}
product_df = pd.DataFrame(product_data)
print(product_df.groupby("Category")["Sales"].sum())
```

#### Solution for Challenge 2:
```python
print(df.groupby(["Region", "Product"])["Sales"].mean())
```

#### Solution for Challenge 3:
```python
print(df.groupby("Region")["Sales"].agg(["sum", "mean"]))
```

---

### Running the Solutions

Let's run each solution cell to verify our expected outputs:

In [7]:
# Solution for Challenge 1:
product_data = {
  "Category": ["Electronics", "Electronics", "Furniture", "Furniture"],
  "Product": ["Phone", "Headphones", "Desk", "Chair"],
  "Sales": [800, 200, 450, 150]
}
product_df = pd.DataFrame(product_data)
print(product_df.groupby("Category")["Sales"].sum())

Category
Electronics    1000
Furniture       600
Name: Sales, dtype: int64


In [8]:
# Solution for Challenge 2:
print(df.groupby(["Region", "Product"])["Sales"].mean())

Region  Product 
North   Keyboard    100.0
        Laptop      500.0
        Monitor     200.0
South   Keyboard    150.0
        Laptop      300.0
        Monitor     250.0
Name: Sales, dtype: float64


In [9]:
# Solution for Challenge 3:
print(df.groupby("Region")["Sales"].agg(["sum", "mean"]))

        sum        mean
Region                 
North   800  266.666667
South   700  233.333333
